# 🤖 Multi-Confirmation Trading Bot
**Strategy:** EMA Crossover + ADX + ATR + Multi-Timeframe Confirmation  
**Supports:** Binance · Kraken · Forex (OANDA)

| Mode | Description |
|---|---|
| `backtest` | Runs on historical data, prints stats + equity chart |
| `paper` | Real market data, virtual money — live dashboard every candle |
| `live` | Real order execution — only after validating paper mode |

**How to use:** Edit `MODE` in the Config cell → `Cell → Run All`

## 📦 Install Dependencies

In [ ]:
!pip install pandas numpy matplotlib python-dotenv ccxt oandapyV20 -q

## ⚙️ Configuration
**This is the only cell you need to edit.**

In [ ]:
import os

# ── Mode ──────────────────────────────────────────────────────
# Options: "backtest" | "paper" | "live"
MODE = os.getenv("MODE", "backtest")

# ── Broker ────────────────────────────────────────────────────
# Options: "binance" | "kraken" | "forex"
BROKER = os.getenv("BROKER", "binance")

# ── Asset ─────────────────────────────────────────────────────
# Binance/Kraken: "BTC/USDT"  "ETH/USDT"  "SOL/USDT"
# Forex (OANDA) : "EUR_USD"   "GBP_USD"   "USD_JPY"
SYMBOL = os.getenv("SYMBOL", "BTC/USDT")

# ── Timeframes ────────────────────────────────────────────────
# Options: "1m" "5m" "15m" "30m" "1h" "4h" "1d"
# HTF must always be one step above LTF
LTF = os.getenv("LTF", "1h")   # Trading timeframe
HTF = os.getenv("HTF", "4h")   # Higher timeframe bias

# ── EMA Crossover ─────────────────────────────────────────────
EMA_FAST = int(os.getenv("EMA_FAST", "9"))
EMA_SLOW = int(os.getenv("EMA_SLOW", "21"))

# ── ADX filter ────────────────────────────────────────────────
ADX_PERIOD    = int(os.getenv("ADX_PERIOD", "14"))
ADX_THRESHOLD = float(os.getenv("ADX_THRESHOLD", "25"))  # Trade only when ADX > this

# ── ATR filter + dynamic SL/TP ────────────────────────────────
ATR_PERIOD  = int(os.getenv("ATR_PERIOD", "14"))
ATR_SL_MULT = float(os.getenv("ATR_SL_MULT", "1.5"))  # SL = entry ± 1.5×ATR
ATR_TP_MULT = float(os.getenv("ATR_TP_MULT", "2.5"))  # TP = entry ± 2.5×ATR

# ── Risk & Capital ────────────────────────────────────────────
RISK_PCT        = float(os.getenv("RISK_PCT", "1.0"))           # % equity per trade
INITIAL_CAPITAL = float(os.getenv("INITIAL_CAPITAL", "10000"))  # Starting capital ($)
BACKTEST_DAYS   = int(os.getenv("BACKTEST_DAYS", "365"))         # Days to backtest

# ── Paper mode ────────────────────────────────────────────────
PAPER_POLL_INTERVAL = int(os.getenv("PAPER_POLL_INTERVAL", "60"))  # Seconds between polls
PAPER_MAX_TRADES    = int(os.getenv("PAPER_MAX_TRADES", "0"))       # 0 = run indefinitely

# ── API Credentials (leave blank for demo/synthetic mode) ─────
BINANCE_API_KEY    = os.getenv("BINANCE_API_KEY", "")
BINANCE_API_SECRET = os.getenv("BINANCE_API_SECRET", "")
KRAKEN_API_KEY     = os.getenv("KRAKEN_API_KEY", "")
KRAKEN_API_SECRET  = os.getenv("KRAKEN_API_SECRET", "")
OANDA_API_KEY      = os.getenv("OANDA_API_KEY", "")
OANDA_ACCOUNT_ID   = os.getenv("OANDA_ACCOUNT_ID", "")
OANDA_ENVIRONMENT  = os.getenv("OANDA_ENVIRONMENT", "practice")  # "practice" | "live"

print(f"✅ Config loaded")
print(f"   Mode    : {MODE.upper()}")
print(f"   Broker  : {BROKER}  |  Symbol : {SYMBOL}")
print(f"   LTF/HTF : {LTF} / {HTF}")
print(f"   Capital : ${INITIAL_CAPITAL:,.0f}  |  Risk per trade: {RISK_PCT}%")

## 📥 Data Fetchers

In [ ]:
import warnings, logging, time
from datetime import datetime, timedelta
from IPython.display import clear_output, display
import pandas as pd
import numpy as np

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s", datefmt="%H:%M:%S")
log = logging.getLogger(__name__)


def fetch_binance(symbol, timeframe, limit=1000):
    import ccxt
    ex = ccxt.binance({"apiKey": BINANCE_API_KEY, "secret": BINANCE_API_SECRET})
    df = pd.DataFrame(ex.fetch_ohlcv(symbol, timeframe, limit=limit),
                      columns=["timestamp","open","high","low","close","volume"])
    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")
    return df.set_index("timestamp")


def fetch_kraken(symbol, timeframe, limit=1000):
    import ccxt
    ex = ccxt.kraken({"apiKey": KRAKEN_API_KEY, "secret": KRAKEN_API_SECRET})
    df = pd.DataFrame(ex.fetch_ohlcv(symbol, timeframe, limit=limit),
                      columns=["timestamp","open","high","low","close","volume"])
    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")
    return df.set_index("timestamp")


def fetch_forex(symbol, timeframe, limit=1000):
    import oandapyV20
    import oandapyV20.endpoints.instruments as instruments
    tf_map = {"1m":"M1","5m":"M5","15m":"M15","30m":"M30","1h":"H1","4h":"H4","1d":"D"}
    client = oandapyV20.API(access_token=OANDA_API_KEY, environment=OANDA_ENVIRONMENT)
    r = instruments.InstrumentsCandles(instrument=symbol,
            params={"granularity": tf_map.get(timeframe,"H1"), "count": limit})
    client.request(r)
    rows = [{"timestamp": pd.to_datetime(c["time"]),
             "open": float(c["mid"]["o"]), "high": float(c["mid"]["h"]),
             "low":  float(c["mid"]["l"]), "close": float(c["mid"]["c"]),
             "volume": int(c["volume"])} for c in r.response["candles"] if c["complete"]]
    return pd.DataFrame(rows).set_index("timestamp")


def fetch_demo_data(symbol, timeframe, days=365):
    """Synthetic OHLCV with realistic trend regimes."""
    tf_m = {"1m":1,"5m":5,"15m":15,"30m":30,"1h":60,"4h":240,"1d":1440}
    mins = tf_m.get(timeframe, 60)
    n = (days * 1440) // mins
    np.random.seed(42)
    lr = np.random.normal(0.0001, 0.002, n)
    for i in range(0, n, n // 6):
        lr[i:i + n//6] += np.random.choice([-1,1]) * 0.0003
    prices = 100 * np.exp(np.cumsum(lr))
    high  = prices * (1 + np.abs(np.random.normal(0, 0.003, n)))
    low   = prices * (1 - np.abs(np.random.normal(0, 0.003, n)))
    open_ = np.roll(prices, 1); open_[0] = prices[0]
    idx   = pd.date_range(end=datetime.now(), periods=n, freq=f"{mins}min")
    return pd.DataFrame({"open":open_,"high":high,"low":low,"close":prices,
                          "volume":np.random.randint(1000,50000,n).astype(float)}, index=idx)


def get_data(symbol, timeframe, days=365):
    tf_m  = {"1m":1,"5m":5,"15m":15,"30m":30,"1h":60,"4h":240,"1d":1440}
    mins  = tf_m.get(timeframe, 60)
    limit = max(500, (days * 1440) // mins)
    creds = {"binance":bool(BINANCE_API_KEY),"kraken":bool(KRAKEN_API_KEY),"forex":bool(OANDA_API_KEY)}
    source = BROKER.upper() if creds.get(BROKER.lower(), False) else "DEMO (synthetic)"
    fetchers = {"binance":fetch_binance,"kraken":fetch_kraken,"forex":fetch_forex}
    df = fetch_demo_data(symbol, timeframe, days) if source.startswith("DEMO") \
         else fetchers[BROKER.lower()](symbol, timeframe, limit)
    candles   = len(df)
    mb_approx = candles * 6 * 8 / 1_048_576   # 6 float64 cols × 8 bytes
    span_days = (df.index[-1] - df.index[0]).days
    print(f"  📥 Data pull   {symbol:<12} {timeframe:<4}  "
          f"source: {source:<18}  "
          f"candles: {candles:>5}  "
          f"span: {span_days}d  "
          f"({df.index[0].strftime('%Y-%m-%d')} → {df.index[-1].strftime('%Y-%m-%d')})  "
          f"~{mb_approx:.3f} MB")
    return df


print("✅ Data fetchers ready")

## 📐 Indicators

In [ ]:
def ema(series, period):
    return series.ewm(span=period, adjust=False).mean()

def calc_atr(df, period=14):
    hl = df["high"] - df["low"]
    hc = (df["high"] - df["close"].shift()).abs()
    lc = (df["low"]  - df["close"].shift()).abs()
    return pd.concat([hl,hc,lc], axis=1).max(axis=1).ewm(span=period, adjust=False).mean()

def calc_adx(df, period=14):
    up, down = df["high"].diff(), -df["low"].diff()
    pdm = np.where((up > down) & (up > 0), up, 0.0)
    mdm = np.where((down > up) & (down > 0), down, 0.0)
    tr  = calc_atr(df, period)
    pdi = 100 * pd.Series(pdm, index=df.index).ewm(span=period, adjust=False).mean() / tr
    mdi = 100 * pd.Series(mdm, index=df.index).ewm(span=period, adjust=False).mean() / tr
    dx  = 100 * (pdi-mdi).abs() / (pdi+mdi).replace(0, np.nan)
    return dx.ewm(span=period, adjust=False).mean()

def compute_indicators(df):
    df = df.copy()
    df["ema_fast"] = ema(df["close"], EMA_FAST)
    df["ema_slow"] = ema(df["close"], EMA_SLOW)
    df["atr"]      = calc_atr(df, ATR_PERIOD)
    df["atr_ma"]   = df["atr"].rolling(ATR_PERIOD * 2).mean()
    df["adx"]      = calc_adx(df, ADX_PERIOD)
    return df.dropna()

print("✅ Indicators ready")

## 🎯 Signal Generation

In [ ]:
def generate_signals(ltf_df, htf_df):
    df = ltf_df.copy()
    htf_trend = (htf_df["ema_fast"] > htf_df["ema_slow"]).astype(int).reindex(df.index, method="ffill")
    cross_up   = (df["ema_fast"] > df["ema_slow"]) & (df["ema_fast"].shift() <= df["ema_slow"].shift())
    cross_down = (df["ema_fast"] < df["ema_slow"]) & (df["ema_fast"].shift() >= df["ema_slow"].shift())
    trend_ok   = df["adx"] > ADX_THRESHOLD
    vol_ok     = df["atr"] > df["atr_ma"]
    df["signal"] = 0
    df.loc[cross_up   & trend_ok & vol_ok & (htf_trend == 1), "signal"] =  1
    df.loc[cross_down & trend_ok & vol_ok & (htf_trend == 0), "signal"] = -1
    sign = df["signal"]
    df["sl"] = np.where(sign ==  1, df["close"] - ATR_SL_MULT * df["atr"],
               np.where(sign == -1, df["close"] + ATR_SL_MULT * df["atr"], np.nan))
    df["tp"] = np.where(sign ==  1, df["close"] + ATR_TP_MULT * df["atr"],
               np.where(sign == -1, df["close"] - ATR_TP_MULT * df["atr"], np.nan))
    return df

print("✅ Signal generator ready")

## 🔁 Backtester

In [ ]:
class Backtester:
    def __init__(self, df):
        self.df = df
        self.equity = INITIAL_CAPITAL
        self.trades = []
        self.equity_curve = []

    def run(self):
        pos = entry_price = sl = tp = entry_time = None
        for ts, row in self.df.iterrows():
            self.equity_curve.append({"timestamp": ts, "equity": self.equity})
            if pos is not None:
                hit_sl = (pos==1 and row["low"]<=sl)  or (pos==-1 and row["high"]>=sl)
                hit_tp = (pos==1 and row["high"]>=tp) or (pos==-1 and row["low"]<=tp)
                if hit_sl or hit_tp:
                    ep  = sl if hit_sl else tp
                    pct = (ep - entry_price) / entry_price * pos
                    rb  = ATR_SL_MULT * row["atr"] / max(entry_price, 1e-9)
                    pnl = self.equity * RISK_PCT / 100 * pct / max(rb, 1e-9)
                    self.equity += pnl
                    self.trades.append({"entry_time":entry_time,"exit_time":ts,
                        "direction":"LONG" if pos==1 else "SHORT",
                        "entry_price":entry_price,"exit_price":ep,
                        "sl":sl,"tp":tp,"pnl":pnl,"result":"WIN" if hit_tp else "LOSS"})
                    pos = None
            if pos is None and row["signal"] != 0:
                pos=row["signal"]; entry_price=row["close"]
                entry_time=ts; sl=row["sl"]; tp=row["tp"]
        return self._stats()

    def _stats(self):
        if not self.trades:
            return {"error": "No trades — try lowering ADX_THRESHOLD or extending BACKTEST_DAYS"}
        df   = pd.DataFrame(self.trades)
        wins = df[df["result"]=="WIN"]; losses = df[df["result"]=="LOSS"]
        eq   = pd.DataFrame(self.equity_curve).set_index("timestamp")
        dd   = (eq["equity"] - eq["equity"].cummax()) / eq["equity"].cummax() * 100
        ret  = df["pnl"] / INITIAL_CAPITAL
        sh   = (ret.mean()/ret.std()*np.sqrt(252)) if ret.std()>0 else 0
        return {
            "total_trades":  len(df),
            "win_rate":      round(len(wins)/len(df)*100,2),
            "total_return":  round((self.equity-INITIAL_CAPITAL)/INITIAL_CAPITAL*100,2),
            "final_equity":  round(self.equity,2),
            "sharpe_ratio":  round(sh,2),
            "max_drawdown":  round(dd.min(),2),
            "avg_win":       round(wins["pnl"].mean(),2) if len(wins)>0 else 0,
            "avg_loss":      round(losses["pnl"].mean(),2) if len(losses)>0 else 0,
            "profit_factor": round(wins["pnl"].sum()/abs(losses["pnl"].sum()),2) if len(losses)>0 else float("inf"),
            "trades_df":     df, "equity_curve": eq,
        }

print("✅ Backtester ready")

## 🧾 Paper Trader
Simulates trades on **real market data** with **virtual money**.  
Prints a live dashboard every candle showing state, all filter results, open position P&L, and a running trade log.

### States printed
| Icon | State | Meaning |
|---|---|---|
| ⏳ | WAITING FOR ENTRY | No position open, monitoring |
| 🟢 | BUY SIGNAL | All filters aligned — entering LONG |
| 🔴 | SELL SIGNAL | All filters aligned — entering SHORT |
| 📈 | HOLDING LONG | In a long, watching SL/TP |
| 📉 | HOLDING SHORT | In a short, watching SL/TP |
| ✅ | EXIT — TAKE PROFIT | TP hit, closing trade |
| ❌ | EXIT — STOP LOSS | SL hit, closing trade |
| 🚫 | SIGNAL BLOCKED | Crossover seen but ADX/ATR filter failed |
| 💤 | NO SIGNAL | No crossover this candle |

In [ ]:
class PaperTrader:
    STATES = {
        "WAIT_ENTRY":  ("⏳", "WAITING FOR ENTRY",  "No position open. Monitoring for signal."),
        "BUY":         ("🟢", "BUY SIGNAL",          "All filters aligned — entering LONG."),
        "SELL":        ("🔴", "SELL SIGNAL",         "All filters aligned — entering SHORT."),
        "HOLD_LONG":   ("📈", "HOLDING LONG",        "In a LONG position. Watching SL / TP."),
        "HOLD_SHORT":  ("📉", "HOLDING SHORT",       "In a SHORT position. Watching SL / TP."),
        "EXIT_TP":     ("✅", "EXIT — TAKE PROFIT",  "TP hit. Trade closed with profit."),
        "EXIT_SL":     ("❌", "EXIT — STOP LOSS",    "SL hit. Trade closed with loss."),
        "FILTER_BLOCK":("🚫", "SIGNAL BLOCKED",      "EMA crossover seen but ADX/ATR filter failed."),
        "NO_SIGNAL":   ("💤", "NO SIGNAL",           "No EMA crossover on this candle."),
    }

    def __init__(self):
        self.equity      = INITIAL_CAPITAL
        self.position    = None       # None | "LONG" | "SHORT"
        self.entry_price = None
        self.entry_time  = None
        self.sl          = None
        self.tp          = None
        self.trade_log   = []
        self.iteration   = 0
        self.state       = "WAIT_ENTRY"

    def _unrealised_pct(self, price):
        if self.position == "LONG":  return (price - self.entry_price) / self.entry_price * 100
        if self.position == "SHORT": return (self.entry_price - price) / self.entry_price * 100
        return 0.0

    def _print_dashboard(self, ts, row, htf_bull, cross_up, cross_down):
        clear_output(wait=True)
        icon, label, desc = self.STATES.get(self.state, ("❓", self.state, ""))
        total_pnl  = self.equity - INITIAL_CAPITAL
        total_pct  = total_pnl / INITIAL_CAPITAL * 100
        wins       = sum(1 for t in self.trade_log if t["result"] == "WIN")
        n_trades   = len(self.trade_log)
        win_rate   = wins / n_trades * 100 if n_trades else 0.0
        unrealised = self._unrealised_pct(row["close"]) if self.position else 0.0

        # Filter results
        adx_ok  = row["adx"]  > ADX_THRESHOLD
        atr_ok  = row["atr"]  > row["atr_ma"]
        htf_ok  = htf_bull if (cross_up or self.position=="LONG") else (not htf_bull)

        W = 56
        s = "═" * W
        d = "─" * W

        print(f"\n{s}")
        print(f"  🤖 PAPER TRADER  │  {BROKER.upper()} │ {SYMBOL} │ {LTF}")
        print(f"  🕐 {ts.strftime('%Y-%m-%d  %H:%M:%S')}   Candle #{self.iteration}")
        print(s)
        print(f"  {icon}  {label}")
        print(f"      {desc}")
        print(d)

        # Market snapshot
        print(f"  📊 MARKET SNAPSHOT")
        print(f"     Price            {row['close']:.5f}")
        print(f"     EMA {EMA_FAST:<2} (fast)      {row['ema_fast']:.5f}")
        print(f"     EMA {EMA_SLOW:<2} (slow)      {row['ema_slow']:.5f}")
        print(f"     ADX              {row['adx']:.2f}")
        print(f"     ATR              {row['atr']:.5f}   MA: {row['atr_ma']:.5f}")
        print(d)

        # Filter checklist
        htf_label = "bullish" if htf_bull else "bearish"
        cross_label = "▲ up" if cross_up else ("▼ down" if cross_down else "— none")
        print(f"  🔍 FILTER CHECKLIST")
        print(f"     {'✅' if htf_bull  else '➡️'}  HTF bias      : {htf_label}  (EMA{EMA_FAST} {'>' if htf_bull else '<'} EMA{EMA_SLOW} on {HTF})")
        print(f"     {'✅' if (cross_up or cross_down) else '❌'}  EMA crossover : {cross_label}")
        print(f"     {'✅' if adx_ok else '❌'}  ADX filter    : {row['adx']:.1f} (need > {ADX_THRESHOLD})")
        print(f"     {'✅' if atr_ok  else '❌'}  ATR filter    : {row['atr']:.5f} {'>' if atr_ok else '<'} MA {row['atr_ma']:.5f}")
        print(d)

        # Open position
        if self.position:
            u_sign = "+" if unrealised >= 0 else ""
            dist_sl = abs(row["close"] - self.sl)
            dist_tp = abs(self.tp - row["close"])
            print(f"  {'📈' if self.position=='LONG' else '📉'} OPEN POSITION")
            print(f"     Direction        {self.position}")
            print(f"     Entry            {self.entry_price:.5f}   @ {self.entry_time.strftime('%d %b %H:%M')}")
            print(f"     Stop Loss        {self.sl:.5f}   ({dist_sl:.5f} away)")
            print(f"     Take Profit      {self.tp:.5f}   ({dist_tp:.5f} away)")
            print(f"     Unrealised PnL   {u_sign}{unrealised:.2f}%")
        else:
            print(f"  💼 OPEN POSITION   None")
        print(d)

        # Account
        eq_sign = "+" if total_pnl >= 0 else ""
        print(f"  💰 ACCOUNT")
        print(f"     Equity           ${self.equity:>10,.2f}")
        print(f"     Total PnL        {eq_sign}${total_pnl:>9,.2f}   ({eq_sign}{total_pct:.2f}%)")
        print(f"     Trades           {n_trades}   Wins: {wins}   Win rate: {win_rate:.1f}%")
        print(d)

        # Recent trades
        if self.trade_log:
            print(f"  📋 RECENT TRADES (last 5)")
            print(f"     {'Date/Time':<16} {'Dir':<6} {'Entry':>8} {'Exit':>8} {'PnL':>10}  Result")
            for t in self.trade_log[-5:]:
                r_icon = "✅ WIN" if t["result"]=="WIN" else "❌ LOSS"
                p      = t["pnl"]
                ps     = f"+${p:.2f}" if p >= 0 else f"-${abs(p):.2f}"
                print(f"     {t['exit_time'].strftime('%m-%d %H:%M'):<16} "
                      f"{t['direction']:<6} {t['entry_price']:>8.4f} "
                      f"{t['exit_price']:>8.4f} {ps:>10}  {r_icon}")
        print(f"{s}\n")

    def _record_exit(self, ts, row, exit_price, hit_tp):
        sign       = 1 if self.position == "LONG" else -1
        pct        = (exit_price - self.entry_price) / self.entry_price * sign
        rb         = ATR_SL_MULT * row["atr"] / max(self.entry_price, 1e-9)
        pnl        = self.equity * RISK_PCT / 100 * pct / max(rb, 1e-9)
        self.equity += pnl
        self.trade_log.append({
            "entry_time":  self.entry_time, "exit_time":   ts,
            "direction":   self.position,
            "entry_price": self.entry_price, "exit_price":  exit_price,
            "sl": self.sl, "tp": self.tp,
            "pnl": pnl, "result": "WIN" if hit_tp else "LOSS",
        })
        self.state       = "EXIT_TP" if hit_tp else "EXIT_SL"
        self.position    = None
        self.entry_price = self.entry_time = self.sl = self.tp = None

    def run(self):
        tf_s = {"1m":60,"5m":300,"15m":900,"30m":1800,"1h":3600,"4h":14400,"1d":86400}
        poll = PAPER_POLL_INTERVAL or tf_s.get(LTF, 3600)
        print(f"🟡 Paper trader started — {BROKER.upper()} | {SYMBOL} | {LTF}")
        print(f"   Virtual capital  : ${INITIAL_CAPITAL:,.2f}")
        print(f"   Poll interval    : {poll}s  |  Press Ctrl+C to stop\n")

        while True:
            try:
                self.iteration += 1
                ltf_raw = get_data(SYMBOL, LTF, days=30)
                htf_raw = get_data(SYMBOL, HTF, days=60)
                ltf_ind = compute_indicators(ltf_raw)
                htf_ind = compute_indicators(htf_raw)

                row      = ltf_ind.iloc[-1]
                prev     = ltf_ind.iloc[-2]
                ts       = ltf_ind.index[-1]
                htf_row  = htf_ind.iloc[-1]
                htf_bull = htf_row["ema_fast"] > htf_row["ema_slow"]

                cross_up   = (row["ema_fast"] > row["ema_slow"]) and (prev["ema_fast"] <= prev["ema_slow"])
                cross_down = (row["ema_fast"] < row["ema_slow"]) and (prev["ema_fast"] >= prev["ema_slow"])
                adx_ok     = row["adx"] > ADX_THRESHOLD
                atr_ok     = row["atr"] > row["atr_ma"]

                # ── Exit check ───────────────────────────────
                if self.position:
                    hit_sl = (self.position=="LONG"  and row["low"]  <= self.sl) or \
                             (self.position=="SHORT" and row["high"] >= self.sl)
                    hit_tp = (self.position=="LONG"  and row["high"] >= self.tp) or \
                             (self.position=="SHORT" and row["low"]  <= self.tp)
                    if hit_sl or hit_tp:
                        self._record_exit(ts, row, self.sl if hit_sl else self.tp, hit_tp)
                        self._print_dashboard(ts, row, htf_bull, cross_up, cross_down)
                        if PAPER_MAX_TRADES and len(self.trade_log) >= PAPER_MAX_TRADES:
                            print(f"✅ Reached max trades ({PAPER_MAX_TRADES}). Stopping.")
                            self._final_summary(); break
                        time.sleep(poll); continue
                    self.state = "HOLD_LONG" if self.position=="LONG" else "HOLD_SHORT"
                    self._print_dashboard(ts, row, htf_bull, cross_up, cross_down)
                    time.sleep(poll); continue

                # ── Entry check ──────────────────────────────
                long_ok  = cross_up   and adx_ok and atr_ok and htf_bull
                short_ok = cross_down and adx_ok and atr_ok and not htf_bull

                if long_ok or short_ok:
                    self.position    = "LONG" if long_ok else "SHORT"
                    self.entry_price = row["close"]
                    self.entry_time  = ts
                    s = 1 if long_ok else -1
                    self.sl = row["close"] - s * ATR_SL_MULT * row["atr"]
                    self.tp = row["close"] + s * ATR_TP_MULT * row["atr"]
                    self.state = "BUY" if long_ok else "SELL"
                elif (cross_up or cross_down) and not (adx_ok and atr_ok):
                    self.state = "FILTER_BLOCK"
                else:
                    self.state = "NO_SIGNAL"

                self._print_dashboard(ts, row, htf_bull, cross_up, cross_down)

                if self.position:
                    self.state = "HOLD_LONG" if self.position=="LONG" else "HOLD_SHORT"

            except KeyboardInterrupt:
                print("\n⛔ Paper trader stopped.")
                self._final_summary(); break
            except Exception as e:
                log.error(f"Loop error: {e}")

            time.sleep(poll)

    def _final_summary(self):
        if not self.trade_log:
            print("No completed trades."); return
        df   = pd.DataFrame(self.trade_log)
        wins = df[df["result"]=="WIN"]
        print(f"\n{'═'*56}")
        print(f"  📊 PAPER TRADING SESSION SUMMARY")
        print(f"     Total trades  : {len(df)}")
        print(f"     Win rate      : {len(wins)/len(df)*100:.1f}%")
        print(f"     Total PnL     : ${df['pnl'].sum():.2f}")
        print(f"     Final equity  : ${self.equity:,.2f}")
        print(f"{'═'*56}\n")
        display(df)


print("✅ PaperTrader ready")

## 🚀 Run — Backtest / Paper / Live
Reads `MODE` from the config cell and routes automatically.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline
plt.style.use("dark_background")


def run_backtest():
    log.info(f"Fetching {SYMBOL} | LTF:{LTF} HTF:{HTF} | {BACKTEST_DAYS}d")
    ltf_raw = get_data(SYMBOL, LTF, days=BACKTEST_DAYS)
    htf_raw = get_data(SYMBOL, HTF, days=BACKTEST_DAYS * 2)
    ltf_ind = compute_indicators(ltf_raw)
    htf_ind = compute_indicators(htf_raw)
    sig_df  = generate_signals(ltf_ind, htf_ind)
    stats   = Backtester(sig_df).run()

    if "error" in stats:
        print(f"⚠️  {stats['error']}"); return

    print(f"""
╔══════════════════════════════════════╗
║         BACKTEST RESULTS             ║
╠══════════════════════════════════════╣
║  Symbol        : {SYMBOL:<20}║
║  Timeframe     : {LTF} → {HTF:<15}║
╠══════════════════════════════════════╣
║  Total trades  : {stats['total_trades']:<20}║
║  Win rate      : {stats['win_rate']:<19}%║
║  Total return  : {stats['total_return']:<19}%║
║  Final equity  : ${stats['final_equity']:<19,}║
║  Sharpe ratio  : {stats['sharpe_ratio']:<20}║
║  Max drawdown  : {stats['max_drawdown']:<19}%║
║  Profit factor : {stats['profit_factor']:<20}║
║  Avg win       : ${stats['avg_win']:<19}║
║  Avg loss      : ${stats['avg_loss']:<19}║
╚══════════════════════════════════════╝
""")

    display(stats["trades_df"].style
        .applymap(lambda v: "color: #1D9E75" if v=="WIN" else "color: #D85A30", subset=["result"])
        .format({"entry_price":"{:.4f}","exit_price":"{:.4f}",
                 "sl":"{:.4f}","tp":"{:.4f}","pnl":"${:.2f}"}))

    trades = stats["trades_df"]; equity = stats["equity_curve"]
    fig = plt.figure(figsize=(16,12), facecolor="#0f0f0f")
    gs  = gridspec.GridSpec(4, 1, figure=fig, hspace=0.06, height_ratios=[3,1,1,1.5])
    ax1 = fig.add_subplot(gs[0])
    ax2 = fig.add_subplot(gs[1], sharex=ax1)
    ax3 = fig.add_subplot(gs[2], sharex=ax1)
    ax4 = fig.add_subplot(gs[3])

    for ax in [ax1,ax2,ax3,ax4]:
        ax.set_facecolor("#0f0f0f")
        ax.tick_params(colors="#666", labelsize=8)
        for sp in ax.spines.values(): sp.set_color("#333")

    ax1.plot(sig_df.index, sig_df["close"],    color="#555",    lw=0.8, label="Price")
    ax1.plot(sig_df.index, sig_df["ema_fast"], color="#378ADD", lw=1.2, label=f"EMA {EMA_FAST}")
    ax1.plot(sig_df.index, sig_df["ema_slow"], color="#E85D24", lw=1.2, label=f"EMA {EMA_SLOW}")
    for _, t in trades.iterrows():
        c = "#1D9E75" if t["direction"]=="LONG" else "#D85A30"
        ax1.scatter(t["entry_time"], t["entry_price"], color=c,     marker="^" if t["direction"]=="LONG" else "v", s=70, zorder=5)
        ax1.scatter(t["exit_time"],  t["exit_price"],  color="#aaa", marker="x", s=40, zorder=5)
    ax1.set_ylabel("Price", color="#888", fontsize=9)
    ax1.legend(loc="upper left", fontsize=8, facecolor="#1a1a1a", labelcolor="white", framealpha=0.7)
    ax1.set_title(f"{SYMBOL} {LTF} | Return:{stats['total_return']}%  WinRate:{stats['win_rate']}%  "
                  f"Sharpe:{stats['sharpe_ratio']}  MaxDD:{stats['max_drawdown']}%",
                  color="white", fontsize=10, pad=8)

    ax2.plot(sig_df.index, sig_df["adx"], color="#9F7FD4", lw=1)
    ax2.axhline(ADX_THRESHOLD, color="#555", lw=0.8, linestyle="--")
    ax2.set_ylabel("ADX", color="#888", fontsize=9); ax2.set_ylim(0,60)

    ax3.plot(sig_df.index, sig_df["atr"],    color="#E8B84B", lw=1,   label="ATR")
    ax3.plot(sig_df.index, sig_df["atr_ma"], color="#555",    lw=0.8, linestyle="--", label="ATR MA")
    ax3.set_ylabel("ATR", color="#888", fontsize=9)
    ax3.legend(loc="upper left", fontsize=7, facecolor="#1a1a1a", labelcolor="white", framealpha=0.6)

    ec = "#1D9E75" if stats["total_return"]>=0 else "#D85A30"
    ax4.plot(equity.index, equity["equity"], color=ec, lw=1.5)
    ax4.fill_between(equity.index, equity["equity"], INITIAL_CAPITAL, alpha=0.15, color=ec)
    ax4.axhline(INITIAL_CAPITAL, color="#555", lw=0.8, linestyle="--")
    ax4.set_ylabel("Equity ($)", color="#888", fontsize=9)
    ax4.set_xlabel("Date", color="#888", fontsize=9)
    plt.tight_layout()
    fname = f"backtest_{SYMBOL.replace('/','-')}_{LTF}.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight", facecolor="#0f0f0f")
    plt.show(); print(f"📁 Chart saved → {fname}")
    csv = f"trades_{SYMBOL.replace('/','-')}_{LTF}.csv"
    stats["trades_df"].to_csv(csv, index=False); print(f"📁 Trade log → {csv}")


# ── Route by MODE ─────────────────────────────────────────────
if MODE == "backtest":
    run_backtest()

elif MODE == "paper":
    PaperTrader().run()

elif MODE == "live":
    print("⚠️  LIVE MODE — real orders will be placed!")
    confirm = input("Type YES to confirm: ")
    if confirm.strip().upper() == "YES":
        class LiveTrader(PaperTrader):
            """Inherits PaperTrader dashboard + logic.
            Add real broker API calls inside _enter_live_order()."""
            def _enter_live_order(self, direction, entry, sl, tp):
                """
                ── Binance / Kraken (ccxt) ──
                exchange.create_order(SYMBOL, 'market', 'buy' if direction=='LONG' else 'sell', qty)
                exchange.create_order(SYMBOL, 'stop',  'sell', qty, sl)
                exchange.create_order(SYMBOL, 'limit', 'sell', qty, tp)

                ── Forex (oandapyV20) ──
                import oandapyV20.endpoints.orders as orders
                data = {"order": {"type":"MARKET","instrument":SYMBOL,
                        "units":"100" if direction=='LONG' else "-100",
                        "stopLossOnFill":{"price":str(sl)},
                        "takeProfitOnFill":{"price":str(tp)}}}
                r = orders.OrderCreate(OANDA_ACCOUNT_ID, data=data)
                client.request(r)
                """
                log.info(f"🔴 LIVE ORDER {direction}  entry={entry:.5f}  SL={sl:.5f}  TP={tp:.5f}")
        LiveTrader().run()
    else:
        print("Cancelled.")

else:
    print(f"❓ Unknown MODE '{MODE}'. Use 'backtest', 'paper', or 'live'.")